In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu124')

print(" Dependencies ready.")

 Dependencies ready.


In [2]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [3]:
import os
for p in os.listdir('/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset'):
    print(p)

kitti_dbinfos_train.pkl
training
kitti_infos_train.pkl
testing
kitti_infos_test.pkl
kitti_infos_val.pkl
gt_database
devkit_object
kitti_infos_trainval.pkl


In [4]:
# ============================================================
# CELL 2 — KITTI Dataset: Mount from Kaggle Datasets
# ============================================================
# The KITTI 3D Object Detection dataset is available on Kaggle at:
#   https://www.kaggle.com/datasets/garymk/kitti-3d-object-detection-dataset
#
# HOW TO ADD IT TO YOUR SESSION (no download needed):
#   1. Open your Kaggle Notebook
#   2. Click the "+" icon next to "Input" on the right panel
#   3. Search "KITTI 3D Object Detection" → Add dataset by garymk
#   4. The data will be mounted at /kaggle/input/kitti-3d-object-detection-dataset/
#
# Expected structure after mounting:
#   /kaggle/input/kitti-3d-object-detection-dataset/
#   └── training/
#       ├── velodyne/     ← .bin files
#       ├── label_2/      ← .txt labels
#       └── calib/        ← .txt calibration
#
# ImageSets (train.txt / val.txt) need to be created if not present:
import os
from pathlib import Path
import urllib.request
import shutil


KITTI_ROOT   = '/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset'
IMAGESET_DIR = '/kaggle/input/datasets/codingyodha/imagesets'
SAVE_DIR     = '/kaggle/working/checkpoints'
CHECKPOINT_INPUT = '/kaggle/input/datasets/shivaprasadbgowda/s2-v4-chkpnt'

os.makedirs(IMAGESET_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,     exist_ok=True)



# Copy all checkpoints from mounted dataset back to working dir
if os.path.exists(CHECKPOINT_INPUT):
    for f in os.listdir(CHECKPOINT_INPUT):
        src = os.path.join(CHECKPOINT_INPUT, f)
        dst = os.path.join(SAVE_DIR, f)
        shutil.copy2(src, dst)
        print(f"  Restored: {f}")
    print(f"Checkpoints restored to {SAVE_DIR}")
else:
    print("No checkpoint dataset found — starting fresh")
# # Official splits hosted by multiple repos — these are canonical
# train_url = 'https://raw.githubusercontent.com/traveller59/second.pytorch/master/second/data/ImageSets/train.txt'
# val_url   = 'https://raw.githubusercontent.com/traveller59/second.pytorch/master/second/data/ImageSets/val.txt'

# urllib.request.urlretrieve(train_url, os.path.join(IMAGESET_DIR, 'train.txt'))
# urllib.request.urlretrieve(val_url,   os.path.join(IMAGESET_DIR, 'val.txt'))

# with open(os.path.join(IMAGESET_DIR, 'train.txt')) as f:
#     n_train = sum(1 for l in f if l.strip())
# with open(os.path.join(IMAGESET_DIR, 'val.txt')) as f:
#     n_val = sum(1 for l in f if l.strip())

# print(f"Official split loaded: {n_train} train / {n_val} val")
# ── Auto-generate ImageSets from available .bin files if not present ──────
train_txt = os.path.join(IMAGESET_DIR, 'train.txt')
val_txt   = os.path.join(IMAGESET_DIR, 'val.txt')

if not os.path.exists(train_txt) or not os.path.exists(val_txt):
    velodyne_dir = os.path.join(KITTI_ROOT, 'training', 'velodyne')
    all_ids = sorted([
        f.replace('.bin', '')
        for f in os.listdir(velodyne_dir) if f.endswith('.bin')
    ])
    split_idx = int(len(all_ids) * (6/7))   # ~87.5% train, ~12.5% val (KITTI standard ratio)
    train_ids = all_ids[:split_idx]
    val_ids   = all_ids[split_idx:]

    with open(train_txt, 'w') as f:
        f.write('\n'.join(train_ids))
    with open(val_txt, 'w') as f:
        f.write('\n'.join(val_ids))

    print(f" ImageSets created: {len(train_ids)} train / {len(val_ids)} val frames")
else:
    with open(train_txt) as f:
        n_train = sum(1 for l in f if l.strip())
    with open(val_txt) as f:
        n_val = sum(1 for l in f if l.strip())
    print(f" ImageSets found: {n_train} train / {n_val} val frames")

print(f"KITTI root : {KITTI_ROOT}")
print(f"Save dir   : {SAVE_DIR}")

  Restored: _rollback.pth
  Restored: train.log
  Restored: gt_database.pkl
  Restored: train_metrics_session_backup.csv
  Restored: session_checkpoint.pth
  Restored: best.pth
  Restored: train_metrics.csv
Checkpoints restored to /kaggle/working/checkpoints
 ImageSets found: 3712 train / 3769 val frames
KITTI root : /kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset
Save dir   : /kaggle/working/checkpoints


In [5]:
# ============================================================
# CELL 3 — dataset.py  (Version 4: 10D, height profile, GT-Aug,
#           full augmentation suite, front-only range)
# ============================================================
import os, pickle
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from pathlib import Path
from typing import Optional

KITTI_CLASSES = {'Car': 0, 'Pedestrian': 1, 'Cyclist': 2}

# ── BEV configuration (Version 4) ────────────────────────────────────────────
# x: front only [0, 70.4m]  — eliminates unlabelled negative-X returns
# y: symmetric  [-40, 40m]
# pillar_size: 70.4/512 ≈ 0.1375m in x,  80/512 ≈ 0.156m in y
# We keep 512x512 grid for WPO compatibility (power-of-2 FFT efficiency)
BEV_CFG = dict(
    x_min=0.0,    x_max=70.4,
    y_min=-40.0,  y_max=40.0,
    z_min=-3.0,   z_max=1.0,
    pillar_size_x=70.4 / 512,   # ≈ 0.1375 m
    pillar_size_y=80.0 / 512,   # ≈ 0.1563 m
    max_points_per_pillar=100,  # up from 32; recovers dense near-range info
    max_pillars=16000,
    bev_h=512,
    bev_w=512,
    input_dim=10,               # x,y,z,r, Δxc,Δyc,Δzc, xp,yp, dist
)

MIN_PILLARS = 8

# GT-Aug sampling counts (from PointPillars paper)
GT_AUG_SAMPLES = {'Car': 15, 'Pedestrian': 2, 'Cyclist': 8}
GT_AUG_MIN_POINTS = {'Car': 5, 'Pedestrian': 5, 'Cyclist': 5}


def points_to_pillars(points: np.ndarray, cfg: dict):
    """
    Convert raw LiDAR point cloud → pillar tensors.

    Input:  points (N, 4) float32  [x, y, z, intensity]
    Output: pillars    (P, max_pts, input_dim)  float32  zero-padded
            num_points (P,)   int32
            coords     (P, 2) int32  [row, col]
            num_pillars int

    Point decoration (10D):
        0-3:  raw  x, y, z, intensity
        4-6:  Δxc, Δyc, Δzc  — offset from pillar point-cloud centroid
        7-8:  xp, yp          — offset from pillar geometric centre
        9:    dist             — Euclidean distance from LiDAR origin
    """
    x_min, x_max = cfg['x_min'], cfg['x_max']
    y_min, y_max = cfg['y_min'], cfg['y_max']
    z_min, z_max = cfg['z_min'], cfg['z_max']
    ps_x, ps_y   = cfg['pillar_size_x'], cfg['pillar_size_y']
    max_pts       = cfg['max_points_per_pillar']
    max_pillars   = cfg['max_pillars']
    H, W          = cfg['bev_h'], cfg['bev_w']
    in_dim        = cfg['input_dim']   # 10

    # ── Range filter ──────────────────────────────────────────────────────────
    mask = (
        (points[:, 0] >= x_min) & (points[:, 0] < x_max) &
        (points[:, 1] >= y_min) & (points[:, 1] < y_max) &
        (points[:, 2] >= z_min) & (points[:, 2] < z_max)
    )
    points = points[mask]

    if len(points) == 0:
        empty = np.zeros((MIN_PILLARS, max_pts, in_dim), dtype=np.float32)
        return empty, np.ones(MIN_PILLARS, dtype=np.int32), \
               np.zeros((MIN_PILLARS, 2), dtype=np.int32), MIN_PILLARS

    # ── Assign points to pillars ──────────────────────────────────────────────
    col_idx   = ((points[:, 0] - x_min) / ps_x).astype(np.int32).clip(0, W - 1)
    row_idx   = ((points[:, 1] - y_min) / ps_y).astype(np.int32).clip(0, H - 1)
    pillar_id = row_idx * W + col_idx

    sort_order = np.argsort(pillar_id, kind='stable')
    sorted_pts = points[sort_order]
    sorted_ids = pillar_id[sort_order]

    unique_ids, first_occ, counts = np.unique(
        sorted_ids, return_index=True, return_counts=True)

    if len(unique_ids) > max_pillars:
        top_k      = np.argpartition(counts, -max_pillars)[-max_pillars:]
        unique_ids = unique_ids[top_k]
        first_occ  = first_occ[top_k]
        counts     = counts[top_k]

    P      = len(unique_ids)
    n_take = np.minimum(counts, max_pts)

    col_range  = np.arange(max_pts, dtype=np.int32)[None, :]
    gather_idx = first_occ[:, None] + np.minimum(col_range, (n_take - 1)[:, None])
    pillars_raw = sorted_pts[gather_idx]              # (P, max_pts, 4)

    valid_mask_np = (col_range < n_take[:, None])     # (P, max_pts) bool
    valid_f       = valid_mask_np[:, :, None].astype(np.float32)

    # ── Geometric features ────────────────────────────────────────────────────
    valid_cnt = n_take[:, None, None].clip(min=1).astype(np.float32)

    # Δxc, Δyc, Δzc: offset from arithmetic mean of all real points in pillar
    sum_xyz   = (pillars_raw[:, :, :3] * valid_f).sum(axis=1, keepdims=True)
    mean_xyz  = sum_xyz / valid_cnt                    # (P, 1, 3)
    delta_c   = (pillars_raw[:, :, :3] - mean_xyz) * valid_f   # (P, max_pts, 3)

    # xp, yp: offset from pillar geometric centre (deterministic, not data-driven)
    pillar_cx = (unique_ids % W).astype(np.float32) * ps_x + x_min + ps_x / 2
    pillar_cy = (unique_ids // W).astype(np.float32) * ps_y + y_min + ps_y / 2
    delta_xp  = (pillars_raw[:, :, 0] - pillar_cx[:, None]) * valid_f[..., 0]
    delta_yp  = (pillars_raw[:, :, 1] - pillar_cy[:, None]) * valid_f[..., 0]

    # dist: Euclidean distance from LiDAR origin
    dist = np.linalg.norm(pillars_raw[:, :, :3], axis=-1, keepdims=True) * valid_f

    # ── Assemble 10D feature vector ───────────────────────────────────────────
    # [x, y, z, r,  Δxc, Δyc, Δzc,  xp, yp,  dist]
    pillars_10d = np.concatenate([
        pillars_raw * valid_f,         # (P, T, 4) raw — zeros where padded
        delta_c,                        # (P, T, 3) centroid offsets
        delta_xp[:, :, None],           # (P, T, 1) pillar-centre offset x
        delta_yp[:, :, None],           # (P, T, 1) pillar-centre offset y
        dist,                           # (P, T, 1) distance
    ], axis=-1).astype(np.float32)      # (P, T, 10)

    num_points = n_take.astype(np.int32)
    coords     = np.stack([unique_ids // W, unique_ids % W], axis=1).astype(np.int32)

    # ── Height profile (μ_z, σ_z, Δz) — pillar-level statistics ─────────────
    # These are scalar per-pillar values, NOT per-point. They will be
    # broadcast across all T slots of each pillar so max-pool preserves them.
    z_vals     = pillars_raw[:, :, 2]                  # (P, T)
    valid_cnt2 = n_take[:, None].clip(min=1).astype(np.float32)
    mu_z       = (z_vals * valid_f[..., 0]).sum(axis=1, keepdims=True) / valid_cnt2[:, 0:1]  # (P,1)
    sq_diff    = ((z_vals - mu_z) ** 2) * valid_f[..., 0]
    sigma_z    = np.sqrt((sq_diff.sum(axis=1, keepdims=True) / valid_cnt2[:, 0:1]).clip(min=0))
    z_max_arr  = (z_vals * valid_f[..., 0] + (-999) * (1 - valid_f[..., 0])).max(axis=1)
    z_min_arr  = (z_vals * valid_f[..., 0] + ( 999) * (1 - valid_f[..., 0])).min(axis=1)
    delta_z    = (z_max_arr - z_min_arr).clip(min=0)[:, None]  # (P,1)

    # Broadcast to (P, T, 3) so max-pool keeps these signals
    mu_z_broad    = np.broadcast_to(mu_z[:, None, :],    (P, max_pts, 1)).copy()
    sigma_z_broad = np.broadcast_to(sigma_z[:, None, :], (P, max_pts, 1)).copy()
    delta_z_broad = np.broadcast_to(delta_z[:, None, :], (P, max_pts, 1)).copy()

    # Zero out padding slots
    mu_z_broad    *= valid_f
    sigma_z_broad *= valid_f
    delta_z_broad *= valid_f

    # Final pillar tensor: 10 + 3 = 13D
    pillars_13d = np.concatenate([
        pillars_10d,       # (P, T, 10)
        mu_z_broad,        # (P, T, 1)  channel 10: μ_z
        sigma_z_broad,     # (P, T, 1)  channel 11: σ_z
        delta_z_broad,     # (P, T, 1)  channel 12: Δz
    ], axis=-1).astype(np.float32)   # → (P, T, 13)

    # ── Minimum pillar guarantee (BN stability) ───────────────────────────────
    if P < MIN_PILLARS:
        n_dup      = MIN_PILLARS - P
        dup_idx    = np.arange(n_dup) % P
        pillars_13d = np.concatenate([pillars_13d, pillars_13d[dup_idx]], axis=0)
        num_points  = np.concatenate([num_points,  num_points[dup_idx]],  axis=0)
        coords      = np.concatenate([coords,      coords[dup_idx]],      axis=0)
        P           = MIN_PILLARS

    return pillars_13d, num_points, coords, P


def parse_kitti_calib(calib_path):
    with open(calib_path, 'r') as f:
        for line in f:
            if line.startswith('Tr_velo_to_cam'):
                vals = list(map(float, line.strip().split(':')[1].split()))
                return np.array(vals, dtype=np.float64).reshape(3, 4)
    raise ValueError(f'Tr_velo_to_cam not found in {calib_path}')


def cam_to_lidar(loc_cam, ry, velo_to_cam):
    R = velo_to_cam[:, :3]
    t = velo_to_cam[:, 3]
    p_lidar = R.T @ (loc_cam - t)
    yaw_lidar = -ry - np.pi / 2
    yaw_lidar = (yaw_lidar + np.pi) % (2 * np.pi) - np.pi
    return float(p_lidar[0]), float(p_lidar[1]), float(p_lidar[2]), float(yaw_lidar)


def parse_kitti_label(label_path, calib_path):
    velo_to_cam = parse_kitti_calib(calib_path)
    objs = []
    with open(label_path, 'r') as f:
        for line in f:
            parts   = line.strip().split()
            cls_str = parts[0]
            if cls_str not in KITTI_CLASSES:
                continue
            h, w, l = float(parts[8]),  float(parts[9]),  float(parts[10])
            x, y, z = float(parts[11]), float(parts[12]), float(parts[13])
            ry      = float(parts[14])
            lx, ly, lz, yaw = cam_to_lidar(
                np.array([x, y, z], dtype=np.float64), ry, velo_to_cam)
            objs.append(dict(class_id=KITTI_CLASSES[cls_str],
                             lx=lx, ly=ly, lz=lz,
                             w=w, l=l, h=h, yaw=yaw))
    return objs


def _check_box_collision(new_box, existing_boxes, margin=0.25):
    """Return True if new_box overlaps with any existing box (simple BEV rect check)."""
    if len(existing_boxes) == 0:
        return False
    cx, cy, _, w, l, _, _ = new_box
    for box in existing_boxes:
        ex, ey, _, ew, el, _, _ = box
        if (abs(cx - ex) < (l + el) / 2 + margin and
                abs(cy - ey) < (w + ew) / 2 + margin):
            return True
    return False


class KITTIPillarDataset(Dataset):
    def __init__(self, root, split='train', imageset_path=None,
                 cfg=BEV_CFG, gt_db_path=None):
        self.root  = Path(root)
        self.cfg   = cfg
        self.split = split
        self.gt_db = None

        if imageset_path is None:
            imageset_path = self.root.parent / 'ImageSets' / f'{split}.txt'
        with open(imageset_path) as f:
            self.frame_ids = [l.strip() for l in f if l.strip()]

        self.lidar_dir = self.root / 'training' / 'velodyne'
        self.label_dir = self.root / 'training' / 'label_2'
        self.calib_dir = self.root / 'training' / 'calib'

        # Load GT database for augmentation (train split only)
        if split == 'train' and gt_db_path and os.path.exists(gt_db_path):
            with open(gt_db_path, 'rb') as f:
                self.gt_db = pickle.load(f)
            print(f'GT database loaded: '
                  f'Car={len(self.gt_db["Car"])}, '
                  f'Ped={len(self.gt_db["Pedestrian"])}, '
                  f'Cyc={len(self.gt_db["Cyclist"])}')

    def __len__(self):
        return len(self.frame_ids)

 

    def _apply_gt_aug_clean(self, points, boxes):
        """Clean implementation of GT-Aug with proper class tracking."""
        if self.gt_db is None:
            return points, boxes

        cfg = self.cfg
        existing_boxes = list(boxes) if len(boxes) > 0 else []
        new_box_rows  = []
        new_points_list = []

        for cls_name, n_sample in GT_AUG_SAMPLES.items():
            cls_id     = KITTI_CLASSES[cls_name]
            db_entries = [e for e in self.gt_db[cls_name]
                          if e['n_points'] >= GT_AUG_MIN_POINTS[cls_name]]
            if not db_entries:
                continue

            idxs = np.random.choice(len(db_entries),
                                    size=min(n_sample, len(db_entries)),
                                    replace=False)
            for idx in idxs:
                entry = db_entries[int(idx)]
                box   = entry['box'].copy()   # [cx,cy,cz,w,l,h,yaw]

                # Per-object jitter
                box[0] += np.random.normal(0, 0.25)
                box[1] += np.random.normal(0, 0.25)
                box[2] += np.random.normal(0, 0.25)
                rot_jitter = np.random.uniform(-np.pi / 20, np.pi / 20)
                box[6] += rot_jitter

                if not (cfg['x_min'] <= box[0] < cfg['x_max'] and
                        cfg['y_min'] <= box[1] < cfg['y_max']):
                    continue

                # Build list of all boxes for collision (existing + already-placed new)
                all_existing = []
                for eb in existing_boxes:
                    all_existing.append(eb[1:] if len(eb) == 8 else eb)
                for nb in new_box_rows:
                    all_existing.append(nb[1:])

                if _check_box_collision(box, all_existing):
                    continue

                # Rotate and translate points
                pts = entry['points_local'].copy()
                cos_j, sin_j = np.cos(rot_jitter), np.sin(rot_jitter)
                pts[:, :2] = pts[:, :2] @ np.array([[cos_j, -sin_j],
                                                      [sin_j,  cos_j]]).T
                pts[:, 0] += box[0]
                pts[:, 1] += box[1]
                pts[:, 2] += box[2]

                new_points_list.append(pts)
                new_box_rows.append(np.array([cls_id, box[0], box[1], box[2],
                                               box[3], box[4], box[5], box[6]],
                                              dtype=np.float32))

        if new_points_list:
            points = np.vstack([points] + new_points_list)
            new_boxes_arr = np.stack(new_box_rows, axis=0)
            if len(boxes) > 0:
                boxes = np.vstack([boxes, new_boxes_arr])
            else:
                boxes = new_boxes_arr

        return points, boxes

    def __getitem__(self, idx):
        fid      = self.frame_ids[idx]
        bin_path = self.lidar_dir / f'{fid}.bin'
        points   = np.fromfile(str(bin_path), dtype=np.float32).reshape(-1, 4)

        boxes = np.zeros((0, 8), dtype=np.float32)
        label_path = self.label_dir / f'{fid}.txt'
        calib_path = self.calib_dir / f'{fid}.txt'
        if label_path.exists() and calib_path.exists():
            objs = parse_kitti_label(str(label_path), str(calib_path))
            if objs:
                boxes = np.array([[o['class_id'], o['lx'], o['ly'], o['lz'],
                                   o['w'], o['l'], o['h'], o['yaw']]
                                  for o in objs], dtype=np.float32)

        if self.split == 'train':
            # ── Step 1: GT-Aug (paste before augmentation so augmentation
            #            applies globally to both original + pasted objects) ──
            if self.gt_db is not None and len(boxes) > 0:
                n_original_boxes = len(boxes)
                points, boxes = self._apply_gt_aug_clean(points, boxes)

            if len(boxes) > 0:
                # ── Step 2: Object-level augmentation ─────────────────────────
                # Each GT box independently jittered (already done inside GT-Aug
                # for pasted objects; apply to ORIGINAL boxes here)
                for bi in range(n_original_boxes):
                    rot_j = np.random.uniform(-np.pi / 20, np.pi / 20)
                    tx    = np.random.normal(0, 0.25)
                    ty    = np.random.normal(0, 0.25)
                    tz    = np.random.normal(0, 0.25)
                    # Jitter box
                    boxes[bi, 1] += tx
                    boxes[bi, 2] += ty
                    boxes[bi, 3] += tz
                    boxes[bi, 7] += rot_j

                # ── Step 3: Global Y-flip ──────────────────────────────────────
                if np.random.rand() < 0.5:
                    points[:, 1]  = -points[:, 1]
                    boxes[:, 2]   = -boxes[:, 2]
                    boxes[:, 7]   = -boxes[:, 7]

                # ── Step 4: Global rotation ────────────────────────────────────
                if np.random.rand() < 0.5:
                    angle = np.random.uniform(-np.pi / 4, np.pi / 4)
                    cos_a, sin_a = np.cos(angle), np.sin(angle)
                    rot  = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float32)
                    points[:, :2]  = points[:, :2] @ rot
                    boxes[:, 1:3]  = boxes[:, 1:3] @ rot
                    boxes[:, 7]   += angle

                # ── Step 5: Global scaling ─────────────────────────────────────
                if np.random.rand() < 0.5:
                    scale = np.random.uniform(0.95, 1.05)
                    points[:, :3] *= scale
                    boxes[:, 1:7] *= scale   # cx, cy, cz, w, l, h all scale

                # ── Step 6: Global translation ─────────────────────────────────
                if np.random.rand() < 0.5:
                    tx = np.random.normal(0, 0.25)
                    ty = np.random.normal(0, 0.25)
                    tz = np.random.normal(0, 0.25)
                    points[:, 0] += tx
                    points[:, 1] += ty
                    points[:, 2] += tz
                    boxes[:, 1]  += tx
                    boxes[:, 2]  += ty
                    boxes[:, 3]  += tz

        pillars, num_pts, coords, n_pil = points_to_pillars(points, self.cfg)

        return {
            'pillars'    : torch.from_numpy(pillars),
            'num_points' : torch.from_numpy(num_pts),
            'coords'     : torch.from_numpy(coords),
            'num_pillars': n_pil,
            'boxes'      : torch.from_numpy(boxes),
            'frame_id'   : fid,
        }


def kitti_collate_fn(batch):
    batch_pillars, batch_numpts, batch_coords = [], [], []
    batch_boxes, frame_ids = [], []
    for b_idx, sample in enumerate(batch):
        P = sample['num_pillars']
        batch_pillars.append(sample['pillars'])
        batch_numpts.append(sample['num_points'])
        b_col = torch.full((P, 1), b_idx, dtype=torch.int32)
        batch_coords.append(torch.cat([b_col, sample['coords']], dim=1))
        batch_boxes.append(sample['boxes'])
        frame_ids.append(sample['frame_id'])
    return {
        'pillars'    : torch.cat(batch_pillars, dim=0),
        'num_points' : torch.cat(batch_numpts,  dim=0),
        'coords'     : torch.cat(batch_coords,  dim=0),
        'boxes'      : batch_boxes,
        'batch_size' : len(batch),
        'frame_ids'  : frame_ids,
    }


def build_dataloaders(kitti_root, imageset_dir, batch_size, num_workers,
                      rank, world_size, cfg=BEV_CFG, gt_db_path=None):
    train_ds = KITTIPillarDataset(kitti_root, 'train',
                                  os.path.join(imageset_dir, 'train.txt'),
                                  cfg, gt_db_path=gt_db_path)
    val_ds   = KITTIPillarDataset(kitti_root, 'val',
                                  os.path.join(imageset_dir, 'val.txt'),
                                  cfg, gt_db_path=None)   # no aug on val

    train_sampler = DistributedSampler(train_ds, num_replicas=world_size,
                                       rank=rank, shuffle=True, drop_last=True)
    val_sampler   = DistributedSampler(val_ds,   num_replicas=world_size,
                                       rank=rank, shuffle=False, drop_last=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=train_sampler, num_workers=num_workers,
                              collate_fn=kitti_collate_fn, pin_memory=True,
                              persistent_workers=(num_workers > 0))
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              sampler=val_sampler,   num_workers=num_workers,
                              collate_fn=kitti_collate_fn, pin_memory=True,
                              persistent_workers=(num_workers > 0))

    return train_loader, val_loader, train_sampler

print("dataset.py v4 loaded — 13D pillars, GT-Aug, full aug suite, front-only range.")

dataset.py v4 loaded — 13D pillars, GT-Aug, full aug suite, front-only range.


In [6]:
# ============================================================
# CELL 3.5 — build_gt_database.py  (run ONCE before training)
# ============================================================
import os, pickle, numpy as np
from pathlib import Path
from tqdm import tqdm

def build_gt_database(kitti_root, imageset_dir, save_dir, split='train'):
    """
    Scans every training frame, extracts the raw LiDAR points that fall
    inside each GT bounding box, and saves them keyed by class.

    Output: SAVE_DIR/gt_database.pkl
    Schema: { 'Car': [ {'points': (N,4), 'box': (7,), 'frame_id': str}, ... ],
              'Pedestrian': [...], 'Cyclist': [...] }
    """
    # from dataset import parse_kitti_label, parse_kitti_calib, KITTI_CLASSES
    # from scipy.spatial.transform import Rotation as R_scipy

    os.makedirs(save_dir, exist_ok=True)
    db_path = os.path.join(save_dir, 'gt_database.pkl')

    if os.path.exists(db_path):
        print(f'GT database already exists at {db_path}. Delete to rebuild.')
        return db_path

    kitti_root = Path(kitti_root)
    imageset_path = os.path.join(imageset_dir, f'{split}.txt')
    with open(imageset_path) as f:
        frame_ids = [l.strip() for l in f if l.strip()]

    lidar_dir = kitti_root / 'training' / 'velodyne'
    label_dir = kitti_root / 'training' / 'label_2'
    calib_dir = kitti_root / 'training' / 'calib'

    database = {'Car': [], 'Pedestrian': [], 'Cyclist': []}
    inv_classes = {v: k for k, v in KITTI_CLASSES.items()}

    def points_in_box(pts, box):
        """Return mask of points inside an axis-aligned then rotated box."""
        cx, cy, cz, w, l, h, yaw = box
        # Translate to box centre
        dx = pts[:, 0] - cx
        dy = pts[:, 1] - cy
        dz = pts[:, 2] - cz
        # Rotate into box frame (undo yaw)
        cos_y, sin_y = np.cos(-yaw), np.sin(-yaw)
        lx = cos_y * dx - sin_y * dy
        ly = sin_y * dx + cos_y * dy
        lz = dz
        # Check within half-extents (+10cm margin for border points)
        margin = 0.1
        mask = (
            (np.abs(lx) <= l / 2 + margin) &
            (np.abs(ly) <= w / 2 + margin) &
            (np.abs(lz) <= h / 2 + margin)
        )
        return mask

    for fid in tqdm(frame_ids, desc='Building GT database'):
        bin_path = lidar_dir / f'{fid}.bin'
        label_path = label_dir / f'{fid}.txt'
        calib_path = calib_dir / f'{fid}.txt'

        if not (bin_path.exists() and label_path.exists() and calib_path.exists()):
            continue

        points = np.fromfile(str(bin_path), dtype=np.float32).reshape(-1, 4)
        objs = parse_kitti_label(str(label_path), str(calib_path))

        for obj in objs:
            cls_name = inv_classes[obj['class_id']]
            box = np.array([obj['lx'], obj['ly'], obj['lz'],
                            obj['w'], obj['l'], obj['h'], obj['yaw']],
                           dtype=np.float32)
            mask = points_in_box(points, box)
            obj_pts = points[mask].copy()

            # Need at least 5 points to be useful
            if len(obj_pts) < 5:
                continue

            # Store points relative to box centre for easy placement later
            obj_pts_local = obj_pts.copy()
            obj_pts_local[:, 0] -= box[0]
            obj_pts_local[:, 1] -= box[1]
            obj_pts_local[:, 2] -= box[2]

            database[cls_name].append({
                'points_local': obj_pts_local,   # (N,4) centred at box origin
                'box': box,                       # (7,) [cx,cy,cz,w,l,h,yaw] world coords
                'n_points': len(obj_pts_local),
                'frame_id': fid,
            })

    # Report stats
    for cls_name, entries in database.items():
        print(f'{cls_name}: {len(entries)} objects in database '
              f'(avg {np.mean([e["n_points"] for e in entries]):.1f} pts)')

    with open(db_path, 'wb') as f:
        pickle.dump(database, f)

    print(f'GT database saved to {db_path}')
    return db_path

# Run it
GT_DB_PATH = build_gt_database(KITTI_ROOT, IMAGESET_DIR, SAVE_DIR)

GT database already exists at /kaggle/working/checkpoints/gt_database.pkl. Delete to rebuild.


In [7]:
# ============================================================
# CELL 4 — model.py  (Multi-Scale BEVWaveFormer)
# ============================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

EPS = 1e-6
_SINC_TAYLOR_THRESH = 0.05
_SPECTRAL_CLAMP = 50.0

class PillarFeatureNet(nn.Module):
    def __init__(self, in_channels: int = 13, out_channels: int = 64, max_points: int = 100):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 32, bias=False),
            nn.BatchNorm1d(32, eps=1e-3),
            nn.SiLU(inplace=True),
            nn.Linear(32, out_channels, bias=False),
            nn.BatchNorm1d(out_channels, eps=1e-3),
            nn.SiLU(inplace=True),
        )
        self.out_channels = out_channels
        self.max_points   = max_points

    def forward(self, pillars: torch.Tensor, num_points_per_pillar: torch.Tensor) -> torch.Tensor:
        P, T, C = pillars.shape
        mask    = (torch.arange(T, device=pillars.device).unsqueeze(0) < num_points_per_pillar.unsqueeze(1))
        mask_f  = mask.unsqueeze(-1).float()

        
        feats  = self.mlp((pillars * mask_f).view(P * T, -1))
        pillar_feats, _ = (feats.view(P, T, self.out_channels) * mask_f).max(dim=1)
        return pillar_feats

class BEVScatter(nn.Module):
    def __init__(self, bev_h: int, bev_w: int):
        super().__init__()
        self.H, self.W = bev_h, bev_w

    def forward(self, pillar_feats: torch.Tensor, coords: torch.Tensor, batch_size: int) -> torch.Tensor:
        C, dtype, device = pillar_feats.shape[1], pillar_feats.dtype, pillar_feats.device
        b_idx = coords[:, 0].long()
        r_idx = coords[:, 1].long().clamp(0, self.H - 1)
        c_idx = coords[:, 2].long().clamp(0, self.W - 1)

        flat_key   = b_idx * (self.H * self.W) + r_idx * self.W + c_idx
        _, inv_idx = torch.unique(flat_key.flip(0), return_inverse=True)
        keep_rev   = torch.zeros(inv_idx.max().item() + 1, dtype=torch.long, device=device).scatter_(0, inv_idx, torch.arange(len(inv_idx), device=device))
        keep_mask  = torch.zeros(len(flat_key), dtype=torch.bool, device=device)
        keep_mask[keep_rev] = True
        keep_mask  = keep_mask.flip(0)

        pillar_feats, b_idx, r_idx, c_idx = pillar_feats[keep_mask], b_idx[keep_mask], r_idx[keep_mask], c_idx[keep_mask]
        flat_idx_exp = (b_idx * (self.H * self.W) + r_idx * self.W + c_idx).unsqueeze(1).expand(-1, C)
        bev_flat = torch.zeros(batch_size * self.H * self.W, C, dtype=dtype, device=device).scatter_(0, flat_idx_exp, pillar_feats)
        return bev_flat.view(batch_size, self.H, self.W, C).permute(0, 3, 1, 2).contiguous()

# FIX 1: _stable_sinc — compute Taylor from raw x (safe for small x),
# exact on full x with clamp only on denominator to avoid division by zero.
def _stable_sinc(omega_d: torch.Tensor, t: float) -> torch.Tensor:
    x      = omega_d * t
    x2     = x * x
    # Taylor series: sin(x)/x ≈ 1 - x²/6 + x⁴/120  (numerically safe for small x)
    taylor = 1.0 - x2 / 6.0 + (x2 * x2) / 120.0
    # Exact: sin(x)/x with denominator guarded; only used where x is large enough
    exact  = torch.sin(x) / x.clamp(min=_SINC_TAYLOR_THRESH)
    return torch.where(x < _SINC_TAYLOR_THRESH, taylor, exact)


class BEVSparsityGating(nn.Module):
    """
    Lightweight density-aware gating applied BEFORE the WPO backbone.

    Why it exists:
    Sparse LiDAR BEV has sharp zero-to-nonzero transitions at pillar boundaries.
    When WPO applies rfft2, these transitions create Gibbs ringing — high-frequency
    spectral spikes that the WPO interprets as real object edges, causing false
    positive detections. This module computes a soft occupancy mask from pillar
    density, then gates the BEV features so empty regions are suppressed (not
    hard-zeroed, which would create new discontinuities) before entering the WPO.

    Implementation:
    1. Compute occupancy: which BEV cells have active pillars (L2 norm > threshold)
    2. Smooth the occupancy mask with a small Gaussian to create soft boundaries
    3. Gate: multiply BEV features by the smoothed mask, then add a small learned
       residual so the WPO still receives some signal in empty regions rather than
       a hard zero (hard zeros are actually WORSE for FFT — they create new spikes)
    """
    def __init__(self, channels: int, smooth_sigma: float = 1.5):
        super().__init__()
        # Learned gating projection: takes smoothed density mask → per-channel gate
        self.gate_proj = nn.Sequential(
            nn.Conv2d(1, channels, 1, bias=True),
            nn.Sigmoid()
        )
        # Gaussian smoothing kernel for the occupancy mask
        k = int(4 * smooth_sigma + 1) | 1   # odd kernel size
        x = torch.arange(k).float() - k // 2
        gauss_1d = torch.exp(-x ** 2 / (2 * smooth_sigma ** 2))
        gauss_2d = (gauss_1d[:, None] * gauss_1d[None, :])
        gauss_2d = gauss_2d / gauss_2d.sum()
        # Register as buffer so it moves to GPU with the model
        self.register_buffer('gauss_kernel',
                             gauss_2d.unsqueeze(0).unsqueeze(0))   # (1,1,k,k)
        self.pad = k // 2

    def forward(self, bev: torch.Tensor) -> torch.Tensor:
        # Occupancy map: 1 where pillar is active, 0 where empty
        # Use channel-wise L2 norm > threshold
        occupancy = (bev.norm(dim=1, keepdim=True) > 0.1).float()   # (B, 1, H, W)

        # Smooth to avoid hard edges creating new FFT discontinuities
        smoothed = torch.nn.functional.conv2d(
            occupancy, self.gauss_kernel,
            padding=self.pad, groups=1
        )   # (B, 1, H, W), values in [0, 1]

        # Per-channel gate: learned mapping from smoothed density to channel weights
        gate = self.gate_proj(smoothed)   # (B, C, H, W)

        # Apply: occupied regions pass full features, empty regions attenuated
        # We do NOT hard-zero empty regions — instead gate ∈ [0, 1] provides
        # smooth attenuation. The small residual (1 - gate) * bev * 0.1 gives
        # the WPO a weak signal in empty areas so it can propagate smoothly
        # rather than hitting a hard zero wall.
        return gate * bev + (1.0 - gate) * bev * 0.1


class WavePropagationOperator(nn.Module):
    def __init__(self, channels: int, bev_h: int, bev_w: int, t: float = 1.0, init_alpha: float = 0.1, init_v: float = 1.0):
        super().__init__()
        self.C, self.H, self.W, self.t = channels, bev_h, bev_w, t
        self.log_alpha = nn.Parameter(torch.tensor(math.log(init_alpha)))
        self.log_v     = nn.Parameter(torch.tensor(math.log(init_v)))

        self.dw_conv  = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.v0_proj  = nn.Sequential(nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(min(8, channels), channels))
        self.out_proj = nn.Sequential(nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(min(8, channels), channels), nn.SiLU(inplace=True))

        wy, wx = torch.meshgrid(torch.fft.fftfreq(bev_h), torch.fft.rfftfreq(bev_w), indexing='ij')
        self.register_buffer('omega2', (wx ** 2 + wy ** 2).unsqueeze(0).unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        U0, V0 = self.dw_conv(x), self.v0_proj(x)
        with torch.amp.autocast('cuda', enabled=False):
            U0_f, V0_f = torch.fft.rfft2(U0.float(), norm='ortho'), torch.fft.rfft2(V0.float(), norm='ortho')
            alpha, v   = torch.clamp(self.log_alpha, -20.0, 20.0).exp().float(), torch.clamp(self.log_v, -20.0, 20.0).exp().float()

            half_a   = alpha / 2.0
            omega_d  = torch.sqrt(torch.clamp((v ** 2) * self.omega2 - half_a ** 2, min=EPS))
            Ut_f     = torch.exp(-half_a * self.t) * (U0_f * torch.cos(omega_d * self.t) + _stable_sinc(omega_d, self.t) * (V0_f + half_a * U0_f))

            # FIX 2: spectral clamp — cap magnitudes at _SPECTRAL_CLAMP without amplifying
            # Old: Ut_f * (_SPECTRAL_CLAMP / clamp(abs, min=_SPECTRAL_CLAMP))  ← amplifies small values
            # New: multiply by ratio only when ratio < 1.0, i.e. only suppress, never amplify
            Ut_f   = Ut_f * torch.clamp(_SPECTRAL_CLAMP / Ut_f.abs().clamp(min=1e-6), max=1.0)
            Ut_f32 = torch.clamp(torch.fft.irfft2(Ut_f, s=(self.H, self.W), norm='ortho'), min=-256.0, max=256.0)

        return torch.clamp(self.out_proj(Ut_f32.to(x.dtype)), min=-65000.0, max=65000.0) + x

class WPOBlock(nn.Module):
    def __init__(self, channels: int, bev_h: int, bev_w: int, expansion: int = 4, use_checkpoint: bool = True):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.norm1 = nn.LayerNorm(channels)
        self.wpo   = WavePropagationOperator(channels, bev_h, bev_w)
        self.norm2 = nn.LayerNorm(channels)
        self.ffn   = nn.Sequential(nn.Linear(channels, channels * expansion), nn.SiLU(inplace=True), nn.Linear(channels * expansion, channels))

    def _inner_forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.wpo(self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2))
        return x + self.ffn(self.norm2(x.permute(0, 2, 3, 1))).permute(0, 3, 1, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_checkpoint and self.training and (x.requires_grad or torch.is_grad_enabled()):
            return checkpoint.checkpoint(self._inner_forward, x, use_reentrant=False)
        return self._inner_forward(x)

class WPOBackbone(nn.Module):
    def __init__(self, in_channels=64, stage_dims=[96, 192, 384, 768], depths=[2, 2, 6, 2], bev_h=512, bev_w=512, use_checkpoint=True):
        super().__init__()
        self.stages, self.downsamples = nn.ModuleList(), nn.ModuleList()
        self.stem = nn.Sequential(nn.Conv2d(in_channels, stage_dims[0], 3, padding=1, bias=False), nn.GroupNorm(min(8, stage_dims[0]), stage_dims[0]), nn.SiLU(inplace=True))
        h, w = bev_h, bev_w
        for i, (dim, depth) in enumerate(zip(stage_dims, depths)):
            self.stages.append(nn.Sequential(*[WPOBlock(dim, h, w, use_checkpoint=use_checkpoint) for _ in range(depth)]))
            if i < len(stage_dims) - 1:
                self.downsamples.append(nn.Sequential(nn.Conv2d(dim, stage_dims[i+1], 2, 2, bias=False), nn.GroupNorm(min(8, stage_dims[i+1]), stage_dims[i+1])))
                h, w = h // 2, w // 2
            else:
                self.downsamples.append(nn.Identity())

    def forward(self, x: torch.Tensor) -> list:
        x, outs = self.stem(x), []
        for stage, down in zip(self.stages, self.downsamples):
            x = stage(x)
            outs.append(x)
            x = down(x)
        return outs

class CenterPointHead(nn.Module):
    REG_DIMS = 8
    def __init__(self, in_channels: int, num_classes: int, hidden: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, hidden), hidden),
            nn.SiLU(inplace=True),
        )
        self.heatmap = nn.Conv2d(hidden, num_classes, 1)
        self.reg     = nn.Conv2d(hidden, self.REG_DIMS, 1)

    def forward(self, feat: torch.Tensor) -> dict:
        h = self.shared(feat)
        return {'heatmap': self.heatmap(h), 'reg': self.reg(h)}

class BEVWaveFormer(nn.Module):
    def __init__(self, pillar_in_ch=13, pillar_out_ch=64, max_points=32, bev_h=512, bev_w=512,
                 stage_dims=[96, 192, 384, 768], depths=[2, 2, 6, 2], use_checkpoint=True):
        super().__init__()

        self.sparsity_gate = BEVSparsityGating(channels=pillar_out_ch)
        self.vfe = PillarFeatureNet(in_channels=13, out_channels=pillar_out_ch,max_points=BEV_CFG['max_points_per_pillar'])
        self.scatter  = BEVScatter(bev_h, bev_w)
        self.backbone = WPOBackbone(pillar_out_ch, stage_dims, depths, bev_h, bev_w, use_checkpoint)

        fpn_dim = 64
        self.fpn_s4_shrink = nn.Sequential(nn.Conv2d(stage_dims[3], fpn_dim, 1, bias=False), nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))
        self.fpn_s4_up     = nn.Sequential(nn.ConvTranspose2d(fpn_dim, fpn_dim, 2, 2), nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.fpn_s3_shrink = nn.Sequential(nn.Conv2d(stage_dims[2], fpn_dim, 1, bias=False), nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))
        self.fpn_s3_up     = nn.Sequential(nn.ConvTranspose2d(fpn_dim, fpn_dim, 2, 2), nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.fpn_s2_shrink = nn.Sequential(nn.Conv2d(stage_dims[1], fpn_dim, 1, bias=False), nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.car_head     = CenterPointHead(in_channels=fpn_dim, num_classes=1)
        self.ped_cyc_head = CenterPointHead(in_channels=fpn_dim, num_classes=2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.eps = 1e-3
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.GroupNorm, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        nn.init.constant_(self.car_head.heatmap.bias, -2.19)
        nn.init.constant_(self.ped_cyc_head.heatmap.bias, -2.19)

    def forward(self, pillars: torch.Tensor, num_points: torch.Tensor, coords: torch.Tensor, batch_size: int) -> dict:
        pillar_feats      = self.vfe(pillars, num_points)
        bev               = self.scatter(pillar_feats, coords, batch_size)
        bev               = self.sparsity_gate(bev)  
        multi_scale_feats = self.backbone(bev)

        s2, s3, s4 = multi_scale_feats[1], multi_scale_feats[2], multi_scale_feats[3]

        f4        = self.fpn_s4_up(self.fpn_s4_shrink(s4))
        fused_128 = f4 + self.fpn_s3_shrink(s3)

        f3        = self.fpn_s3_up(fused_128)
        fused_256 = f3 + self.fpn_s2_shrink(s2)

        return {
            'car': self.car_head(fused_128),
            'ped_cyc': self.ped_cyc_head(fused_256)
        }

print("model.py cell loaded (Fixed: _stable_sinc + spectral clamp).")

model.py cell loaded (Fixed: _stable_sinc + spectral clamp).


In [8]:
# ============================================================
# CELL 5 — train.py  (Multi-Scale Targets, DIoU, & Checkpointing)
# ============================================================
import os, io, logging, math, time, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast

# ===========================================================================
# Logging
# ===========================================================================
def get_logger(rank: int, log_file: str = 'train.log') -> logging.Logger:
    logger = logging.getLogger('BEVWaveFormer')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    if rank == 0:
        fmt = logging.Formatter('[%(asctime)s][%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
        ch = logging.StreamHandler(); ch.setFormatter(fmt); logger.addHandler(ch)
        fh = logging.FileHandler(log_file); fh.setFormatter(fmt); logger.addHandler(fh)
    else:
        logger.addHandler(logging.NullHandler())
    return logger

# ===========================================================================
# Gaussian & Target Generation (Multi-Scale)
# FIX: Pedestrians and Cyclists use 2*yaw encoding (sin/cos of 2θ) to resolve
# the π-periodicity ambiguity in KITTI annotations — their bodies are nearly
# symmetric so the network cannot distinguish yaw from yaw+π using 1θ encoding.
# Cars use standard 1θ encoding (unambiguous from shape asymmetry).
# ===========================================================================
def gaussian_radius(det_size: tuple, min_overlap: float = 0.7) -> float:
    h, w = det_size
    a1=1.0; b1=h+w; c1=h*w*(1-min_overlap)/(1+min_overlap); r1=(b1-math.sqrt(max(b1**2-4*a1*c1,0)))/(2*a1)
    a2=4.0; b2=2*(h+w); c2=(1-min_overlap)*h*w; r2=(b2-math.sqrt(max(b2**2-4*a2*c2,0)))/(2*a2)
    a3=4*min_overlap; b3=-2*min_overlap*(h+w); c3=(min_overlap-1)*h*w; r3=(b3+math.sqrt(max(b3**2-4*a3*c3,0)))/(2*a3)
    return max(0.0, min(r1, r2, r3))

def draw_gaussian(heatmap: torch.Tensor, center_x: int, center_y: int, radius: int) -> None:
    H, W, sigma, diameter = heatmap.shape[0], heatmap.shape[1], max(radius/3.0, 1e-2), 2*radius+1
    x = torch.arange(diameter, dtype=torch.float32) - radius
    g2d = torch.exp(-x**2/(2*sigma**2))[:, None] * torch.exp(-x**2/(2*sigma**2))[None, :]
    x0, y0, x1, y1 = center_x-radius, center_y-radius, center_x+radius+1, center_y+radius+1
    hm_x0, hm_x1, hm_y0, hm_y1 = max(x0,0), min(x1,W), max(y0,0), min(y1,H)
    if hm_x0 >= hm_x1 or hm_y0 >= hm_y1: return
    patch = g2d[hm_y0-y0:hm_y1-y0, hm_x0-x0:hm_x1-x0].to(heatmap.device)
    heatmap[hm_y0:hm_y1, hm_x0:hm_x1] = torch.maximum(heatmap[hm_y0:hm_y1, hm_x0:hm_x1], patch)

def build_gt_targets_multiscale(batch_boxes, batch_size, bev_cfg, device) -> dict:
    car_hm   = torch.zeros(batch_size, 1, 128, 128, device=device)
    car_reg  = torch.zeros(batch_size, 8, 128, 128, device=device)
    car_mask = torch.zeros(batch_size, 1, 128, 128, device=device)

    ped_hm   = torch.zeros(batch_size, 2, 256, 256, device=device)
    ped_reg  = torch.zeros(batch_size, 8, 256, 256, device=device)
    ped_mask = torch.zeros(batch_size, 1, 256, 256, device=device)

    x_min, x_max, y_min, y_max = bev_cfg['x_min'], bev_cfg['x_max'], bev_cfg['y_min'], bev_cfg['y_max']

    for b_idx, boxes in enumerate(batch_boxes):
        if boxes is None or len(boxes) == 0: continue
        for box in (boxes.cpu() if isinstance(boxes, torch.Tensor) else boxes):
            cls_id = int(box[0])
            bev_x, bev_y, lz = float(box[1]), float(box[2]), float(box[3])
            bw, bl, bh, yaw  = float(box[4]), float(box[5]), float(box[6]), float(box[7])

            if not (x_min <= bev_x < x_max and y_min <= bev_y < y_max): continue

            if cls_id == 0:
                grid_size, grid_c_id = 128, 0
                hm, reg, mask = car_hm, car_reg, car_mask
            else:
                grid_size, grid_c_id = 256, cls_id - 1
                hm, reg, mask = ped_hm, ped_reg, ped_mask

            cell_size = (x_max - x_min) / grid_size
            exact_px = (bev_x - x_min) / cell_size
            exact_py = (bev_y - y_min) / cell_size
            px = max(0, min(int(exact_px), grid_size - 1))
            py = max(0, min(int(exact_py), grid_size - 1))

            r_x, r_y = max(1, int(bl/(2*cell_size))), max(1, int(bw/(2*cell_size)))
            draw_gaussian(hm[b_idx, grid_c_id], px, py, max(0, int(gaussian_radius((r_y*2, r_x*2)))))

            reg[b_idx, 0, py, px] = exact_px - px
            reg[b_idx, 1, py, px] = exact_py - py
            reg[b_idx, 2, py, px] = lz
            reg[b_idx, 3, py, px] = math.log(max(bw, 1e-3))
            reg[b_idx, 4, py, px] = math.log(max(bl, 1e-3))
            reg[b_idx, 5, py, px] = math.log(max(bh, 1e-3))

            # FIX: Class-aware yaw encoding to resolve π-periodicity ambiguity.
            # Cars:        encode yaw directly — shape asymmetry disambiguates front/back.
            # Ped/Cyclist: encode 2*yaw — body symmetry makes yaw and yaw+π look identical
            #              to the network. 2θ encoding maps the π period onto a full 2π
            #              sin/cos cycle, giving the network an unambiguous training signal.
            #              Decoder must divide atan2 result by 2 (done in Cell 8).
            if cls_id == 0:
                # Car: standard 1θ encoding
                reg[b_idx, 6, py, px] = math.sin(yaw)
                reg[b_idx, 7, py, px] = math.cos(yaw)
            else:
                # Ped/Cyclist: 2θ encoding
                reg[b_idx, 6, py, px] = math.sin(2.0 * yaw)
                reg[b_idx, 7, py, px] = math.cos(2.0 * yaw)

            mask[b_idx, 0, py, px] = 1.0

    return {
        'car':     {'heatmap': car_hm, 'reg': car_reg, 'reg_mask': car_mask, 'cell_size': (x_max - x_min)/128},
        'ped_cyc': {'heatmap': ped_hm, 'reg': ped_reg, 'reg_mask': ped_mask, 'cell_size': (x_max - x_min)/256}
    }

# ===========================================================================
# DIoU Loss
# ===========================================================================
class BEVLoss(nn.Module):
    def __init__(self, alpha: float = 2.0, beta: float = 4.0, reg_weight: float = 2.0):
        super().__init__()
        self.alpha, self.beta, self.reg_weight = alpha, beta, reg_weight

    def focal_loss(self, pred, target):
        pred = torch.clamp(pred.float().sigmoid(), min=1e-4, max=1-1e-4)
        pos_mask, neg_mask = (target == 1).float(), (target < 1).float()
        pos_loss = torch.log(pred) * (1 - pred) ** self.alpha * pos_mask
        neg_loss = torch.log(1 - pred) * pred ** self.alpha * (1 - target) ** self.beta * neg_mask
        return -(pos_loss.sum() + neg_loss.sum()) / pos_mask.sum().clamp(min=1)

    def _head_loss(self, pred_dict, tgt_dict, cell_size):
        hm_loss = self.focal_loss(pred_dict['heatmap'], tgt_dict['heatmap'])
        mask    = tgt_dict['reg_mask'].float()
        num_pos = mask.sum().clamp(min=1)

        p_reg, t_reg = pred_dict['reg'].float() * mask, tgt_dict['reg'].float() * mask

        l1_loss = F.smooth_l1_loss(p_reg[:, 2:], t_reg[:, 2:], reduction='sum') / num_pos

        dx_p, dy_p = p_reg[:, 0], p_reg[:, 1]
        dx_t, dy_t = t_reg[:, 0], t_reg[:, 1]
        dist_sq_phys = ((dx_p - dx_t) * cell_size)**2 + ((dy_p - dy_t) * cell_size)**2
        diag_sq = torch.exp(t_reg[:, 3])**2 + torch.exp(t_reg[:, 4])**2

        diou_penalty = (dist_sq_phys / torch.clamp(diag_sq, min=1e-3)).sum() / num_pos
        return hm_loss + self.reg_weight * (l1_loss + diou_penalty)

    def forward(self, preds: dict, targets: dict) -> torch.Tensor:
        loss_car = self._head_loss(preds['car'],     targets['car'],     targets['car']['cell_size'])
        loss_ped = self._head_loss(preds['ped_cyc'], targets['ped_cyc'], targets['ped_cyc']['cell_size'])
        return loss_car + loss_ped

# ===========================================================================
# Checkpointing & Session Timer
# ===========================================================================
def save_checkpoint(model, optimizer, epoch, loss, save_dir, rank, filename=None):
    if rank != 0: return
    os.makedirs(save_dir, exist_ok=True)
    if filename is None: filename = f'ckpt_epoch{epoch:04d}.pth'
    path = os.path.join(save_dir, filename)
    state = {'epoch': epoch, 'model_state': model.module.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}
    tmp_path = path + '.tmp'
    torch.save(state, tmp_path)
    os.replace(tmp_path, path)
    latest_path = os.path.join(save_dir, 'latest.pth')
    if os.path.islink(latest_path) or os.path.exists(latest_path): os.remove(latest_path)
    os.symlink(os.path.abspath(path), latest_path)

def load_checkpoint(model, optimizer, checkpoint_path, device, logger) -> int:
    logger.info(f'Loading checkpoint: {checkpoint_path}')
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state'])
    model.to(device)
    optimizer.load_state_dict(ckpt['optimizer_state'])
    logger.info(f'Resumed from epoch {ckpt["epoch"]}, loss={ckpt.get("loss", float("nan")):.4f}')
    return ckpt['epoch'] + 1

class SessionTimer:
    def __init__(self, limit_hours: float = 11.0):
        self.limit_seconds = limit_hours * 3600
        self.start_time    = time.time()
    def elapsed(self) -> float: return time.time() - self.start_time
    def should_save_and_exit(self) -> bool: return self.elapsed() >= self.limit_seconds
    def remaining_str(self) -> str:
        rem = max(0.0, self.limit_seconds - self.elapsed())
        h, r = divmod(int(rem), 3600); m, s = divmod(r, 60)
        return f'{h:02d}h{m:02d}m{s:02d}s'

def save_session_checkpoint(model, optimizer, scheduler, epoch, val_loss, save_dir, rank, logger, patience_counter=0, best_val_loss=float('inf')):
    if rank != 0: return
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, 'session_checkpoint.pth')
    state = {
        'epoch': epoch, 
        'model_state': model.module.state_dict(), 
        'optimizer_state': optimizer.state_dict(), 
        'scheduler_state': scheduler.state_dict(), 
        'loss': val_loss,
        'patience_counter': patience_counter, 
        'best_val_loss': best_val_loss,
    }
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)
    logger.info(f'Session checkpoint saved → {path}  (epoch {epoch})')

    csv_src = os.path.join(save_dir, 'train_metrics.csv')
    csv_dst = os.path.join(save_dir, 'train_metrics_session_backup.csv')
    if os.path.exists(csv_src):
        shutil.copy2(csv_src, csv_dst)
        logger.info(f'Metrics CSV backed up → {csv_dst}')

# ===========================================================================
# Training & Validation Loops
# ===========================================================================
def train_one_epoch(model, loader, optimizer, criterion, device, epoch, logger,
                    bev_cfg, save_dir, log_interval=50):
    model.train()
    total_loss, skipped, max_norm, t0 = 0.0, 0, 2.0, time.time()
    safe_ckpt_path = os.path.join(save_dir, '_rollback.pth')
    if dist.get_rank() == 0:
        torch.save({'model': model.module.state_dict(),'optimizer': optimizer.state_dict()}, safe_ckpt_path)
    dist.barrier()

    
    for step, batch in enumerate(loader):
        pillars, num_points, coords = batch['pillars'].to(device), batch['num_points'].to(device), batch['coords'].to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", dtype=torch.bfloat16):
            preds   = model(pillars, num_points, coords, batch['batch_size'])
            targets = build_gt_targets_multiscale(batch['boxes'], batch['batch_size'], bev_cfg, device)
            loss    = criterion(preds, targets)

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)

        if not torch.isfinite(grad_norm) or any(not torch.isfinite(p).all() for p in model.parameters()):
            optimizer.zero_grad(set_to_none=True)
            logger.warning(f'Epoch {epoch} Step {step}: NaN detected. Rolling back.')
            dist.barrier()
            ckpt_safe = torch.load(safe_ckpt_path, map_location=device)
            model.module.load_state_dict(ckpt_safe['model']); optimizer.load_state_dict(ckpt_safe['optimizer'])
            skipped += 1
            continue

        if epoch < 5:
            for name, param in model.named_parameters():
                if ('log_alpha' in name or 'log_v' in name) and param.grad is not None:
                    param.grad.zero_()

        optimizer.step()

        with torch.no_grad():
            for name, param in model.named_parameters():
                if 'log_alpha' in name or 'log_v' in name:
                    param.clamp_(-20.0, 20.0)

        if step > 0 and step % 50 == 0:
            if dist.get_rank() == 0:
                torch.save({'model': model.module.state_dict(),'optimizer': optimizer.state_dict()}, safe_ckpt_path)
            dist.barrier()
        total_loss += loss.item()

        if step % log_interval == 0:
            logger.info(f'Epoch {epoch:04d} | Step {step:05d} | loss={loss.item():.4f} | grad_norm={grad_norm:.3f}')

            if dist.get_rank() == 0:
                csv_path = os.path.join(save_dir, 'train_metrics.csv')
                if not os.path.exists(csv_path):
                    with open(csv_path, 'w') as f:
                        f.write("Epoch,Step,TrainLoss,GradNorm,MaxNorm\n")
                with open(csv_path, 'a') as f:
                    f.write(f"{epoch},{step},{loss.item():.4f},{grad_norm:.3f},{max_norm:.3f}\n")

    return total_loss / max(len(loader) - skipped, 1)

@torch.no_grad()
def validate(model, loader, criterion, device, epoch, logger, bev_cfg):
    model.eval()
    total_loss, count = torch.tensor(0.0, device=device), torch.tensor(0, device=device)
    for batch in loader:
        with autocast("cuda", dtype=torch.bfloat16):
            preds = model(batch['pillars'].to(device), batch['num_points'].to(device), batch['coords'].to(device), batch['batch_size'])
            targets = build_gt_targets_multiscale(batch['boxes'], batch['batch_size'], bev_cfg, device)
            total_loss += criterion(preds, targets)
            count += 1

    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(count,      op=dist.ReduceOp.SUM)
    avg_loss = (total_loss / count).item()
    logger.info(f'Epoch {epoch:04d} | VAL loss={avg_loss:.4f}')
    return avg_loss

print("train.py loaded (Fixed: 2θ yaw encoding for Pedestrian/Cyclist).")
print("train.py cell loaded (Fixed: explicit bev_cfg + save_dir params).")

train.py loaded (Fixed: 2θ yaw encoding for Pedestrian/Cyclist).
train.py cell loaded (Fixed: explicit bev_cfg + save_dir params).


In [9]:
# ============================================================
# CELL 6 — validate_checkpoint.py  (Multi-Scale Adaptations)
# ============================================================
import os, math
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast

CLASS_NAMES = ['Car', 'Pedestrian', 'Cyclist']

class ValMetricsMultiscale:
    def __init__(self):
        self.C = 3
        self.loss_sum = 0.0; self.n_batches = 0
        self.gt_response = [0.0] * self.C
        self.gt_count    = [0]   * self.C
        self.recall_03   = [0]   * self.C
        self.recall_05   = [0]   * self.C

    @torch.no_grad()
    def update(self, preds, targets, criterion):
        self.loss_sum += criterion(preds, targets).item()
        self.n_batches += 1

        pred_prob_car = preds['car']['heatmap'].float().sigmoid()
        tgt_hm_car    = targets['car']['heatmap']
        self._eval_class(0, pred_prob_car[:, 0], tgt_hm_car[:, 0])

        pred_prob_ped = preds['ped_cyc']['heatmap'].float().sigmoid()
        tgt_hm_ped    = targets['ped_cyc']['heatmap']
        self._eval_class(1, pred_prob_ped[:, 0], tgt_hm_ped[:, 0])
        self._eval_class(2, pred_prob_ped[:, 1], tgt_hm_ped[:, 1])

    def _eval_class(self, c_idx, pred_prob, tgt_hm):
        centres = (tgt_hm == 1.0)
        if centres.any():
            resp = pred_prob[centres]
            self.gt_response[c_idx] += resp.sum().item()
            self.gt_count[c_idx]    += centres.sum().item()
            self.recall_03[c_idx]   += (resp >= 0.3).sum().item()
            self.recall_05[c_idx]   += (resp >= 0.5).sum().item()

    def aggregate(self, device):
        def _r(val):
            t = torch.tensor(float(val), device=device)
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
            return t.item()
        self.loss_sum  = _r(self.loss_sum)
        self.n_batches = int(_r(self.n_batches))
        for c in range(self.C):
            self.gt_response[c] = _r(self.gt_response[c])
            self.gt_count[c]    = int(_r(self.gt_count[c]))
            self.recall_03[c]   = int(_r(self.recall_03[c]))
            self.recall_05[c]   = int(_r(self.recall_05[c]))

    def report(self, logger, epoch=None, save_dir=None):
        avg_loss = self.loss_sum / max(self.n_batches, 1)
        logger.info('=' * 62)
        logger.info(f'  Val total loss : {avg_loss:.4f} (Focal + DIoU Reg)')
        logger.info('-' * 62)
        logger.info(f'  {"Class":<14} {"GT objs":>8} {"AvgResp":>9} {"Recall@.3":>10} {"Recall@.5":>10}')
        logger.info('-' * 62)

        results = {}
        for c, name in enumerate(CLASS_NAMES):
            n        = max(self.gt_count[c], 1)
            avg_resp = self.gt_response[c] / n
            rec03    = self.recall_03[c]   / n
            rec05    = self.recall_05[c]   / n
            logger.info(f'  {name:<14} {self.gt_count[c]:>8d} {avg_resp:>9.4f} {rec03:>10.4f} {rec05:>10.4f}')
            results[name] = dict(recall_at_03=rec03, recall_at_05=rec05)
        logger.info('=' * 62)

        if dist.get_rank() == 0 and save_dir:
            csv_path = os.path.join(save_dir, 'val_metrics.csv')
            write_h = not os.path.exists(csv_path)
            with open(csv_path, 'a') as f:
                if write_h: f.write("Epoch,TotalLoss,Car_R3,Car_R5,Ped_R3,Cyc_R3\n")
                f.write(f"{epoch if epoch else '?'},{avg_loss:.4f},{results['Car']['recall_at_03']:.4f},{results['Car']['recall_at_05']:.4f},{results['Pedestrian']['recall_at_03']:.4f},{results['Cyclist']['recall_at_03']:.4f}\n")
        return results

# FIX 3 (continued): All notebook globals are now explicit parameters
def run_validation(checkpoint_path, kitti_root, imageset_dir,
                   bev_cfg, save_dir,
                   batch_size=2, num_workers=4, split='val',
                   rank=0, local_rank=0, world_size=1, device=None):

    if device is None: device = torch.device(f'cuda:{local_rank}')

    logger = get_logger(rank, log_file=os.path.join(save_dir, 'val.log'))

    if rank == 0: logger.info(f'Validation | {world_size} GPU(s) | ckpt={checkpoint_path}')

    model = BEVWaveFormer(
        pillar_in_ch=13, pillar_out_ch=64, max_points=bev_cfg['max_points_per_pillar'],
        bev_h=bev_cfg['bev_h'], bev_w=bev_cfg['bev_w'],
        stage_dims=[96, 192, 384, 768], depths=[2, 2, 6, 2], use_checkpoint=False
    )

    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(ckpt.get('model_state', ckpt), strict=False)
    if rank == 0: logger.info(f'Checkpoint loaded. epoch={ckpt.get("epoch", "?")}')

    model = model.to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)
    model.eval()

    ds      = KITTIPillarDataset(kitti_root, split, os.path.join(imageset_dir, f'{split}.txt'), bev_cfg)
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    loader  = DataLoader(ds, batch_size=batch_size, sampler=sampler,
                         num_workers=num_workers, collate_fn=kitti_collate_fn, pin_memory=True)

    metrics   = ValMetricsMultiscale()
    criterion = BEVLoss(reg_weight=2.0).to(device)

    with torch.no_grad():
        for batch in loader:
            pillars, num_points, coords, bs = (batch['pillars'].to(device), batch['num_points'].to(device),
                                               batch['coords'].to(device), batch['batch_size'])
            with autocast('cuda', dtype=torch.bfloat16):
                preds = model(pillars, num_points, coords, bs)
            targets = build_gt_targets_multiscale(batch['boxes'], bs, bev_cfg, device)
            metrics.update(preds, targets, criterion)

    metrics.aggregate(device)
    if rank == 0:
        metrics.report(logger, epoch=ckpt.get('epoch', None), save_dir=save_dir)
        logger.info('Validation done.')

    return metrics

print("validate_checkpoint.py cell loaded (Fixed: all globals now explicit params).")

validate_checkpoint.py cell loaded (Fixed: all globals now explicit params).


In [10]:
# ============================================================
# CELL 7 — Main training entry point
# ============================================================
from torch.nn.parallel import DistributedDataParallel as DDP
CFG = dict(
    kitti_root       = KITTI_ROOT,
    imageset_dir     = IMAGESET_DIR,
    save_dir         = SAVE_DIR,
    checkpoint_path  = CHECKPOINT_INPUT,
    resume           = None,
    epochs           = 200,
    batch_size       = 18,
    lr               = 1e-4,
    weight_decay     = 0.01,
    num_workers      = 4,
    log_interval     = 50,
    save_interval    = 5,
    session_hours    = 11.0,
    gt_db_path       = os.path.join(SAVE_DIR, 'gt_database.pkl'),
)

_session_ckpt = os.path.join(CFG['checkpoint_path'], 'session_checkpoint.pth')
if CFG['resume'] is None and os.path.exists(_session_ckpt):
    print(f"  Found session_checkpoint.pth — auto-resuming from {_session_ckpt}")
    CFG['resume'] = _session_ckpt

import torch.distributed as dist_mod

if not dist_mod.is_initialized():
    os.environ.setdefault('MASTER_ADDR', 'localhost')
    os.environ.setdefault('MASTER_PORT', '29500')
    os.environ.setdefault('RANK',        '0')
    os.environ.setdefault('WORLD_SIZE',  '1')
    os.environ.setdefault('LOCAL_RANK',  '0')
    dist_mod.init_process_group(backend='nccl', init_method='env://')

rank       = dist_mod.get_rank()
world_size = dist_mod.get_world_size()
local_rank = int(os.environ.get('LOCAL_RANK', 0))

torch.cuda.set_device(local_rank)
device = torch.device(f'cuda:{local_rank}')

os.makedirs(CFG['save_dir'], exist_ok=True)
logger = get_logger(rank, log_file=os.path.join(CFG['save_dir'], 'train.log'))

if rank == 0:
    logger.info(f'World size     : {world_size}  |  Device : {device}')
    logger.info(f'Per-rank batch : {CFG["batch_size"]}  |  Global batch : {CFG["batch_size"] * world_size}')
    logger.info(f'AMP dtype      : bfloat16')
    logger.info(f'Session limit  : {CFG["session_hours"]} hours')

model = BEVWaveFormer(
    pillar_in_ch  = 13,
    pillar_out_ch = 64,
    max_points    = BEV_CFG['max_points_per_pillar'],
    bev_h         = BEV_CFG['bev_h'],
    bev_w         = BEV_CFG['bev_w'],
    stage_dims    = [96, 192, 384, 768],
    depths        = [2, 2, 6, 2],
    use_checkpoint= True,
).to(device)

model = DDP(model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
            gradient_as_bucket_view=True)

if rank == 0:
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f'Model parameters: {n_params:.1f} M')

spectral_params = [p for n, p in model.named_parameters() if 'log_alpha' in n or 'log_v' in n]
other_params    = [p for n, p in model.named_parameters() if 'log_alpha' not in n and 'log_v' not in n]

optimizer = torch.optim.AdamW([
    {'params': other_params,    'lr': CFG['lr']},
    {'params': spectral_params, 'lr': CFG['lr'] * 0.01, 'weight_decay': 0.0},
], weight_decay=CFG['weight_decay'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 1e-2)

criterion = BEVLoss(reg_weight=2.0).to(device)

train_loader, val_loader, train_sampler = build_dataloaders(
    CFG['kitti_root'], CFG['imageset_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    rank=rank, world_size=world_size,
    gt_db_path=CFG['gt_db_path'],
)

start_epoch = 0
if CFG['resume']:
    if os.path.exists(CFG['resume']):
        ckpt = torch.load(CFG['resume'], map_location='cpu')
        model.module.load_state_dict(ckpt['model_state'])
        model.module.to(device)
        optimizer.load_state_dict(ckpt['optimizer_state'])
        if 'scheduler_state' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        patience_counter = ckpt.get('patience_counter', 0)   # add here
        best_val_loss    = ckpt.get('best_val_loss', float('inf'))
        logger.info(f'Resumed from epoch {ckpt["epoch"]}, loss={ckpt.get("loss", float("nan")):.4f}')
        if dist_mod.is_initialized():
            dist_mod.barrier()
    else:
        logger.warning(f'Resume path not found: {CFG["resume"]} — starting fresh.')

session_timer = SessionTimer(limit_hours=CFG['session_hours'])
val_loss      = float('inf')
best_val_loss = float('inf')

if rank == 0:
    logger.info(f'Starting training from epoch {start_epoch} to epoch {CFG["epochs"]-1}')

import gc
torch.cuda.empty_cache(); gc.collect()
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved  = torch.cuda.memory_reserved(0)  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM : {total:.2f} GB")
print(f"Allocated  : {allocated:.2f} GB")
print(f"Reserved   : {reserved:.2f} GB")
print(f"Free       : {total - reserved:.2f} GB")

# Early stopping
best_val_loss    = float('inf')
patience         = 35          # stop if no improvement for 20 epochs
patience_counter = 0           # restored from checkpoint if resuming

# # Restore patience counter from checkpoint if present
# if CFG['resume'] and os.path.exists(CFG['resume']):
#     ckpt = torch.load(CFG['resume'], map_location='cpu')
#     patience_counter = ckpt.get('patience_counter', 0)
    
#     best_val_loss    = ckpt.get('best_val_loss', float('inf'))



for epoch in range(start_epoch, CFG['epochs']):
    train_sampler.set_epoch(epoch)

    # FIX 3: pass bev_cfg and save_dir explicitly
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion,
        device, epoch, logger,
        bev_cfg=BEV_CFG, save_dir=CFG['save_dir'],
        log_interval=CFG['log_interval'])

    val_loss = validate(
        model, val_loader, criterion, device, epoch, logger,
        bev_cfg=BEV_CFG)

    scheduler.step()

    logger.info(
        f'─── Epoch {epoch:04d} | train={train_loss:.4f} '
        f'| val={val_loss:.4f} '
        f'| lr={scheduler.get_last_lr()[0]:.2e} '
        f'| session_remaining={session_timer.remaining_str()} ───')

    if (epoch + 1) % CFG['save_interval'] == 0:
        save_checkpoint(model, optimizer, epoch, val_loss, CFG['save_dir'], rank)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        save_checkpoint(model, optimizer, epoch, val_loss,
                        CFG['save_dir'], rank, filename='best.pth')
        logger.info(f'  ↳ New best val loss: {val_loss:.4f}')
    else:
        patience_counter += 1
        logger.info(f'  ↳ No improvement. Patience: {patience_counter}/{patience}')

    if patience_counter >= patience:
        logger.info(f'Early stopping triggered at epoch {epoch}.')
        save_session_checkpoint(model, optimizer, scheduler, epoch,val_loss, CFG['save_dir'], rank, logger)
        break

    if session_timer.should_save_and_exit():
        logger.info(f'Session time limit reached after epoch {epoch}. Saving and stopping.')
        save_session_checkpoint(model, optimizer, scheduler, epoch,
                                val_loss, CFG['save_dir'], rank, logger)
        save_checkpoint(model, optimizer, epoch, val_loss,
                        CFG['save_dir'], rank,
                        filename=f'ckpt_epoch{epoch:04d}_session_end.pth')
        if dist_mod.is_initialized():
            dist_mod.barrier()
        break

    if dist_mod.is_initialized():
        dist_mod.barrier()

else:
    save_checkpoint(model, optimizer, CFG['epochs'] - 1, val_loss,
                    CFG['save_dir'], rank, filename='final.pth')
    logger.info('Training complete — all epochs finished.')

if dist_mod.is_initialized():
    dist_mod.destroy_process_group()
    print("Process group destroyed.")

  Found session_checkpoint.pth — auto-resuming from /kaggle/input/datasets/shivaprasadbgowda/s2-v4-chkpnt/session_checkpoint.pth


[2026-04-06 15:04:07][INFO] World size     : 1  |  Device : cuda:0
[2026-04-06 15:04:07][INFO] Per-rank batch : 18  |  Global batch : 18
[2026-04-06 15:04:07][INFO] AMP dtype      : bfloat16
[2026-04-06 15:04:07][INFO] Session limit  : 11.0 hours
[2026-04-06 15:04:11][INFO] Model parameters: 23.7 M
[2026-04-06 15:04:11][INFO] Resumed from epoch 97, loss=5.4726
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[2026-04-06 15:04:11][INFO] Starting training from epoch 98 to epoch 199


GT database loaded: Car=13308, Ped=2079, Cyc=701
GPU        : NVIDIA RTX PRO 6000 Blackwell Server Edition
Total VRAM : 94.97 GB
Allocated  : 0.35 GB
Reserved   : 0.37 GB
Free       : 94.60 GB


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [768, 768, 1, 1], strides() = [768, 1, 768, 768]
bucket_view.sizes() = [768, 768, 1, 1], strides() = [768, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2026-04-06 15:04:27][INFO] Epoch 0098 | Step 00000 | loss=2.3793 | grad_norm=123.647
[2026-04-06 15:07:03][INFO] Epoch 0098 | Step 00050 | loss=2.7294 | grad_norm=200.813
[2026-04-06 15:09:38][INFO] Epoch 0098 | Step 00100 | loss=2.8814 | grad_norm=230.585
[2026-04-06 15:12:13][INFO] Epoch 0098 | Step 00150 | loss=3.0059 | grad_norm=142.896
[2026-04-06 15:1

Process group destroyed.


In [11]:
# ============================================================
# CELL 8 — inference.py  (Center-Pooling NMS & Box Decoding)
# ============================================================
import torch
import torch.nn.functional as F

def decode_head_predictions(hm, reg, bev_cfg, grid_size, class_id_offset, top_k=100):
    """
    Decodes predictions from a single head using per-class 3x3 Center-Pooling.

    Yaw decoding is class-aware to match the encoding in build_gt_targets_multiscale:
      - Car head  (class_id_offset=0): yaw = atan2(sin, cos)          [1θ encoding]
      - Ped/Cyc   (class_id_offset=1): yaw = atan2(sin, cos) / 2      [2θ encoding]
        The 2θ scheme resolves the π-periodicity ambiguity for symmetric bodies.
    """
    B, C, H, W = hm.shape
    device = hm.device

    hm = hm.sigmoid()
    # Per-channel peak suppression: max_pool2d on [B, C, H, W] is spatially
    # local and channel-independent, so Car peaks never suppress Ped/Cyc peaks.
    peak_mask = (hm == F.max_pool2d(hm, kernel_size=3, stride=1, padding=1))
    hm = hm * peak_mask

    hm_flat = hm.view(B, -1)
    top_scores, top_idx = torch.topk(hm_flat, k=min(top_k, hm_flat.shape[1]), dim=1)

    top_c = (top_idx // (H * W)).long()
    top_y = ((top_idx % (H * W)) // W).long()
    top_x = ((top_idx % (H * W)) % W).long()

    b_idx = torch.arange(B, device=device).unsqueeze(1).expand_as(top_x)

    dx    = reg[b_idx, 0, top_y, top_x]
    dy    = reg[b_idx, 1, top_y, top_x]
    z     = reg[b_idx, 2, top_y, top_x]
    log_w = reg[b_idx, 3, top_y, top_x]
    log_l = reg[b_idx, 4, top_y, top_x]
    log_h = reg[b_idx, 5, top_y, top_x]
    sin_v = reg[b_idx, 6, top_y, top_x]
    cos_v = reg[b_idx, 7, top_y, top_x]

    x_min, y_min = bev_cfg['x_min'], bev_cfg['y_min']
    pixel_size   = (bev_cfg['x_max'] - bev_cfg['x_min']) / grid_size

    abs_x = x_min + (top_x.float() + dx) * pixel_size
    abs_y = y_min + (top_y.float() + dy) * pixel_size

    w   = torch.exp(log_w)
    l   = torch.exp(log_l)
    h   = torch.exp(log_h)

    # FIX: match the class-aware encoding from build_gt_targets_multiscale.
    # Car head uses 1θ: yaw = atan2(sin(yaw), cos(yaw))
    # Ped/Cyc head uses 2θ: encoded as sin(2*yaw), cos(2*yaw)
    #   → decode:  angle_2theta = atan2(sin2, cos2)
    #              yaw = angle_2theta / 2
    # The /2 halves the range to (-π/2, π/2], which covers all physical
    # orientations for symmetric pedestrians and cyclists.
    if class_id_offset == 0:
        # Car: 1θ — full [-π, π] range
        yaw = torch.atan2(sin_v, cos_v)
    else:
        # Ped/Cyclist: 2θ — recover yaw from doubled-angle encoding
        yaw = torch.atan2(sin_v, cos_v) / 2.0

    true_class_id = top_c + class_id_offset

    # Output: [B, K, 9] → (x, y, z, w, l, h, yaw, score, class_id)
    return torch.stack([abs_x, abs_y, z, w, l, h, yaw, top_scores,
                        true_class_id.float()], dim=-1)


def inference_step(model, pillars, num_points, coords, batch_size, bev_cfg, top_k=100):
    """End-to-end inference through the Dual-Scale Heads."""
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            preds = model(pillars, num_points, coords, batch_size)

    # Car head (128×128) | class offset = 0 | 1θ yaw decoding
    car_boxes = decode_head_predictions(
        preds['car']['heatmap'].float(),
        preds['car']['reg'].float(),
        bev_cfg, grid_size=128, class_id_offset=0, top_k=top_k
    )

    # Ped/Cyc head (256×256) | class offset = 1 | 2θ yaw decoding
    ped_cyc_boxes = decode_head_predictions(
        preds['ped_cyc']['heatmap'].float(),
        preds['ped_cyc']['reg'].float(),
        bev_cfg, grid_size=256, class_id_offset=1, top_k=top_k
    )

    final_boxes = torch.cat([car_boxes, ped_cyc_boxes], dim=1)

    sorted_boxes = []
    for b in range(batch_size):
        b_boxes = final_boxes[b]
        b_boxes = b_boxes[b_boxes[:, 7].argsort(descending=True)][:top_k]
        sorted_boxes.append(b_boxes)

    return torch.stack(sorted_boxes, dim=0)


print("inference.py loaded (Fixed: 2θ yaw decode for Pedestrian/Cyclist).")

inference.py loaded (Fixed: 2θ yaw decode for Pedestrian/Cyclist).
